# Nami robot playground

Use the environment containing the **bundled Neuromeka wheel**. Edit the robot IP below.
Motion examples are opt-in. Enabling the gripper initializes it during setup.

Nami is one controller with **22 joint values** and **three task chains**:

| State | Layout |
| --- | --- |
| `q[0:2]` | torso (`Joint_L0`, `Joint_L1`) |
| `q[2:4]` | head |
| `q[4:11]` | left arm |
| `q[11:18]` | right arm |
| `q[18:22]` | four dummy slots |
| `p[0:6]` | head, `arm_index=0` |
| `p[6:12]` | left TCP, `arm_index=1` |
| `p[12:18]` | right TCP, `arm_index=2` |

Joints use degrees; task poses use `[x, y, z, rx, ry, rz]` in mm and xyz Euler degrees.
Servo, stop, recovery, direct teaching, and teleoperation mode affect the whole controller.


In [1]:
from pathlib import Path
import sys
import time

# Start Jupyter anywhere inside this repository.
deploy_dir = next(
    (candidate for root in (Path.cwd(), *Path.cwd().parents)
     for candidate in (root, root / "deploy")
     if (candidate / "communication" / "robot.py").is_file()), None)
if deploy_dir is None:
    raise RuntimeError("Start Jupyter inside the neuromeka-il repository")
if str(deploy_dir) not in sys.path:
    sys.path.insert(0, str(deploy_dir))

from communication.robot import NamiRobot
from data_collector.etc.nami_teleop import NamiTeleop, NamiTeleopConfig
from helper.extra_utils import ROBOT_STATE


In [2]:
ROBOT_IP = "192.168.0.67"
ARM_INDICES = (1, 2)  # (1,) for only the left arm; (2,) for only the right.
# Enable neither, either, or both grippers. Physical tool indices may differ.
gripper_config = {
    "per_arm": {
        1: {"enable": False, "type": "EndportDHGripperClient",
            "params": {"tool_index": 0, "speed": 50, "force": 50}},  # left
        2: {"enable": False, "type": "EndportDHGripperClient",
            "params": {"tool_index": 1, "speed": 50, "force": 50}},  # right
    },
}
robot = NamiRobot(robot_ip=ROBOT_IP, gripper_config=gripper_config)


In [14]:
robot.set_servo_all(True)

{'code': '0', 'msg': ''}

In [ ]:
# 0: unlock all joints
# 1~: lock the corresponding joint
robot.robot_client.set_locked_joint(2)

{'code': '0', 'msg': ''}

In [9]:
robot.robot_client.get_control_state().keys()

dict_keys(['q', 'qdot', 'qddot', 'qdes', 'qdotdes', 'qddotdes', 'p', 'pdot', 'pddot', 'pdes', 'pdotdes', 'pddotdes', 'tau', 'tau_act', 'tau_ext', 'tau_jts', 'tau_jts_raw1', 'tau_jts_raw2', 'manipulability'])

In [12]:
robot.robot_client.set_direct_teaching(False)

{'msg': 'SetDirectTeaching Success', 'code': '0'}

## Read state
The adapter preserves the complete controller state. Use `get_task_pose` to select a TCP.


In [4]:
state = robot.get_state()
{key: state[key] for key in ("q", "qdot", "p", "pdot", "op_state")}


{'q': [0.04772866,
  0.010157973,
  24.896635,
  -0.0082941335,
  -137.04091,
  -25.287655,
  73.769585,
  -102.11064,
  74.59095,
  -38.286613,
  0.25941467,
  137.00049,
  24.999996,
  -73.00041,
  101.999405,
  -73.99948,
  37.999702,
  0.0005218506,
  0.0,
  0.0,
  0.0,
  0.0],
 'qdot': [0.0,
  0.00319579,
  -0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  -0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  -0.0,
  -0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'p': [52.478935,
  0.043760106,
  621.4189,
  -0.003851137,
  24.90641,
  0.03858405,
  350.97412,
  451.55475,
  370.23563,
  20.065895,
  -178.62541,
  69.55915,
  354.12918,
  -449.9217,
  373.11584,
  158.86359,
  -1.7611852,
  110.11115],
 'pdot': [0.02496128,
  2.0793308e-05,
  -0.0057159746,
  -4.626224e-07,
  0.00319579,
  1.956857e-19,
  0.01095102,
  9.122446e-06,
  -0.022386132,
  -0.0029927692,
  0.0010260532,
  -0.00045124447,
  0.01111167,
  9.256271e-06,
  -0.022520227,
  0.0030004305,
  0.0009893237,
  0.00048138286],
 '

In [5]:
{arm: robot.get_task_pose(state, arm) for arm in (0, 1, 2)}


{0: [52.60015, -0.00475186, 621.3429, -179.99997, 155.00056, 179.99478],
 1: [351.34772, 451.22662, 370.2988, 20.073496, -178.62785, 69.50537],
 2: [353.67563, -450.2558, 373.1853, 158.85475, -1.7563475, 110.05784]}

## Controller kinematics
Read-only forward and inverse kinematics queries using the robot controller.


In [ ]:
state = robot.get_state()
robot.compute_forward_kinematics(state["q"], arm_index=1)


In [ ]:
robot.compute_inverse_kinematics(
    robot.get_task_pose(state, 1), state["q"], arm_index=1)


## Optional robot commands
Uncomment individual commands when ready. Refresh the state before changing a target.


In [ ]:
# robot.set_servo_all(enable=True)
# robot.recover()
# robot.set_direct_teaching(enable=True)
# robot.set_direct_teaching(enable=False)

# q = list(robot.get_state()["q"])
# q[4] += 2.0
# robot.move(q, mode="joint_abs", vel_ratio=10, acc_ratio=10, wait=True)

# p = robot.get_task_pose(robot.get_state(), 1)
# p[2] += 2.0
# robot.move(p, mode="task_abs", arm_index=1, vel_ratio=10, acc_ratio=10, wait=True)

# robot.stop_teleop()
# robot.stop_motion(mode="smooth")


## Optional DH gripper control
Enable the left (`1`), right (`2`), or both entries in `gripper_config["per_arm"]`
and set each physical `tool_index`. Rerun setup to initialize enabled grippers.
Leave both disabled for no grippers. Uncomment individual commands below.
Values are `1.0` (open), `0.0` (closed), or any value in between.


In [ ]:
# Left gripper:
# robot.move_gripper(mode="no_thread", value=1.0, arm_index=1)  # open
# robot.move_gripper(mode="no_thread", value=0.0, arm_index=1)  # close

# Right gripper:
# robot.move_gripper(mode="no_thread", value=1.0, arm_index=2)  # open
# robot.move_gripper(mode="no_thread", value=0.0, arm_index=2)  # close

# Half-open all enabled grippers:
# robot.move_gripper(mode="no_thread", value=0.5)

# Independent targets with both enabled, ordered left then right:
# robot.move_gripper(mode="no_thread", value=[1.0, 0.0])

# Reinitialize an enabled gripper after a controller restart:
# robot.gripper_clients[1].initialize()


## Stream a fixed task target
Set `RUN_FIXED_TARGET=True` to stream the current TCP poses with `movetelel_abs`.
The helper starts from idle and stops in `finally`.


In [ ]:
def start_teleop(timeout=10.0):
    robot.stop_teleop()
    deadline = time.monotonic() + timeout
    while robot.get_state()["op_state"] != ROBOT_STATE.IDLE:
        if time.monotonic() >= deadline:
            raise TimeoutError("Controller did not become idle")
        time.sleep(0.05)
    while True:
        robot.start_teleop(mode="task_abs")
        time.sleep(0.2)
        op_state = robot.get_state()["op_state"]
        if op_state == ROBOT_STATE.TELE_OP:
            return
        if ROBOT_STATE.in_failure_state(op_state):
            raise RuntimeError(f"Controller failure: {op_state}")
        if time.monotonic() >= deadline:
            raise TimeoutError("Controller did not enter teleoperation")

RUN_FIXED_TARGET = False
if RUN_FIXED_TARGET:
    session = NamiTeleop(robot, NamiTeleopConfig(arm_indices=ARM_INDICES))
    try:
        start_teleop()
        state = robot.get_state()
        session.reset(state)
        targets = session.device_poses(state)
        for _ in range(20):
            started = time.monotonic()
            command = session.convert(targets, robot.get_state())
            for arm_index, pose in command.items():
                robot.tele_move(pose, mode="task_abs", arm_index=arm_index, vel_scale=0.1, acc_scale=0.5)
            time.sleep(max(0, 0.05 - (time.monotonic() - started)))
    finally:
        robot.stop_teleop()
